# Solution 2.2.2 — Data Types and Cleaning

### Path Setup

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../../data/0_raw'
FILE_NAME = 'datania_households_raw.csv'
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

df = pd.read_csv(raw_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df.head(5)

,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw,survey_date,pop_density,education_code,age
0,HH0001,01,Eastern Province,Kurtosis Bay,Urban,4,45 000,2025-01-10,780,3,42
1,HH0002,02,Northern Province,Vector Hills,Rural,6,Ar 32000,2025-01-11,120,2,39
2,HH0003,03,Central Province,Polaris District,Urban,3,54000,2025-01-12,640,4,33
3,HH0004,04,Southern Province,Lagoon Point,Rural,5,NaN,2025-01-13,80,1,51
4,HH0005,05,Western Province,Gamma Plains,Urban,2,unknown,2025-01-13,520,2,28


---

## Task 1 — DataFrame vs Series

In [2]:
print(type(df))
print(type(df['income_dkw']))

<class 'pandas.DataFrame'>
<class 'pandas.Series'>


In [3]:
df.dtypes

hh_id               str
region_code         str
province_name       str
district            str
urban_rural         str
hh_size           int64
income_dkw          str
survey_date         str
pop_density       int64
education_code    int64
age               int64
dtype: object

**Answers:**

- `income_dkw` is `str` (text), not numeric. pandas falls back to text when any value in the column cannot be parsed as a number — here because of values like `'Ar 32000'`, `'unknown'`, and `'45 000'`.
- `survey_date` is `str`. Before extracting the month you need to convert it with `pd.to_datetime()` — only then does the `.dt` accessor become available.
- `hh_id` and `region_code` are `str` intentionally — we forced it with `dtype={'hh_id': str, 'region_code': str}` at load time. Without that argument, `region_code` would be inferred as `int64`, silently dropping leading zeros (`'01'` → `1`).

---

## Task 2 — Fix categorical columns: `.replace()` vs `.str.replace()`

In [4]:
print('urban_rural:', df['urban_rural'].value_counts(dropna=False))
print()
print('region_code:', df['region_code'].value_counts(dropna=False))

urban_rural: urban_rural
Urban    14
Rural    13
Urbn      1
Name: count, dtype: int64

region_code: region_code
05     5
06     5
02     4
03     4
04     4
01     3
1      1
NaN    1
99     1
Name: count, dtype: int64


In [5]:
df['urban_rural'] = df['urban_rural'].replace('Urbn', 'Urban')

print(df['urban_rural'].value_counts(dropna=False))

urban_rural
Urban    15
Rural    13
Name: count, dtype: int64


In [6]:
df['region_code'] = df['region_code'].replace('99', np.nan)
df['region_code'] = df['region_code'].replace('1', '01')

print(df['region_code'].value_counts(dropna=False))

region_code
05     5
06     5
01     4
02     4
03     4
04     4
NaN    2
Name: count, dtype: int64


**Answers:**

- `.str.replace('1', '01')` replaces every occurrence of the character `'1'` inside each string, not the whole value. So `'01'` → `'001'`, `'21'` → `'021'`, `'16'` → `'016'` — all wrong. `.replace()` with a dict or scalar does exact-value matching, so only the string `'1'` (the whole value) is affected.
- After the fix there are two sources of `NaN` in `region_code`: the original `NaN` from rows that had no region recorded, and the row where `'99'` was replaced with `np.nan`.
- `.str.replace()` is the right tool when you need to remove a substring from inside a string — for example stripping `'Ar'` or commas from `income_dkw`, or standardising whitespace inside a label.

---

## Task 3 — Clean `income_dkw`: from messy text to a numeric column

**Step 1: Inspect before writing any code.**

In [7]:
df['income_dkw'].unique()

<ArrowStringArray>
[   '45 000',  'Ar 32000',     '54000',         nan,   'unknown',     '-5000',
    '120000',     '38000', '1,200,000',   ' 41000 ',     '29000',     '65000',
     '87000', 'Ar 74 000',     '51000',     '26000',     '94000',     '33000',
     '56000',    '999999',     '35000',  'Ar 99000',     '61000',     '58000',
     '39000']
Length: 25, dtype: str

**Step 2: Build the cleaning chain interactively.**

In [8]:
income_text = df['income_dkw'].astype('string')

income_text = income_text.str.replace(' ', '', regex=False)
income_text = income_text.str.replace('Ar', '', regex=False)
income_text = income_text.str.replace(',', '', regex=False)
income_text = income_text.replace('unknown', np.nan)
income_text = income_text.replace('NA', np.nan)

income_text.unique()

<ArrowStringArray>
[  '45000',   '32000',   '54000',      <NA>,   '-5000',  '120000',   '38000',
 '1200000',   '41000',   '29000',   '65000',   '87000',   '74000',   '51000',
   '26000',   '94000',   '33000',   '56000',  '999999',   '35000',   '99000',
   '61000',   '58000',   '39000']
Length: 24, dtype: string

**Step 3: Collapse into a single chain and store as a new column.**

In [9]:
df['income_dkw_clean'] = (
    df['income_dkw']
    .astype('string')
    .str.replace(' ', '', regex=False)
    .str.replace('Ar', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace('unknown', np.nan)
    .replace('NA', np.nan)
)

df[['income_dkw', 'income_dkw_clean']].head(15)

,income_dkw,income_dkw_clean
0,45 000,45000
1,Ar 32000,32000
2,54000,54000
3,NaN,<NA>
4,unknown,<NA>
5,-5000,-5000
6,120000,120000
7,38000,38000
8,"1,200,000",1200000
9,41000,41000


**Step 4: Wrap the logic in a reusable function and apply it.**

In [10]:
def clean_income(value, chars_to_remove=None, null_codes=None):
    """Clean a messy income string: remove formatting and replace invalid codes with NaN."""
    if chars_to_remove is None:
        chars_to_remove = [' ', 'Ar', ',']
    if null_codes is None:
        null_codes = ['unknown', 'NA', 'nan', 'not recorded']

    text = str(value)
    for char in chars_to_remove:
        text = text.replace(char, '')
    if text in null_codes:
        return np.nan
    return text


print(clean_income('Ar 32,000'))    # → '32000'
print(clean_income('45 000'))       # → '45000'
print(clean_income('unknown'))      # → nan
print(clean_income(np.nan))         # → nan

32000
45000
nan
nan


In [ ]:
df['income_dkw_clean'] = df['income_dkw'].apply(lambda x: clean_income(x))

df['income_dkw_num'] = pd.to_numeric(df['income_dkw_num'], errors='coerce')

df[['income_dkw', 'income_dkw_clean', 'income_dkw_num']].head(15)

,income_dkw,income_dkw_clean,income_dkw_num
0,45 000,45000,"45,000.00"
1,Ar 32000,32000,"32,000.00"
2,54000,54000,"54,000.00"
3,NaN,NaN,NaN
4,unknown,NaN,NaN
5,-5000,-5000,"-5,000.00"
6,120000,120000,"120,000.00"
7,38000,38000,"38,000.00"
8,"1,200,000",1200000,"1,200,000.00"
9,41000,41000,"41,000.00"


In [12]:
df[df['income_dkw_num'].isna()][['hh_id', 'income_dkw', 'income_dkw_clean']]

,hh_id,income_dkw,income_dkw_clean
3,HH0004,NaN,NaN
4,HH0005,unknown,NaN
16,HH0016,NaN,NaN
21,HH0021,NaN,NaN


**Answers:**

- Rows still `NaN` after conversion are those whose raw value was `'unknown'`, `'NA'`, or already empty — these became `NaN` during the text cleaning step. Any value that the cleaning pipeline could not reduce to a plain number also ends up as `NaN` via `errors='coerce'`.
- `.str.replace()` replaces a substring within every string in the Series — it operates character by character. `.replace()` (without `.str`) matches and replaces whole values — `'unknown'` is only replaced when the entire cell value equals `'unknown'`, not when it appears as part of a longer string.
- Testing on individual values first lets you verify corner cases (empty string, `NaN`, mixed formats) without applying bad logic to 5,000 rows. Once `.apply()` runs you have to re-inspect the whole column to find what went wrong.

---

## Task 4 — Convert `survey_date` to datetime

In [13]:
df['survey_date'].unique()

<ArrowStringArray>
[  '2025-01-10',   '2025-01-11',   '2025-01-12',   '2025-01-13',
   '2025-01-14',   '03/15/2025',   '2025-01-16',   '2025-01-17',
   '2025/01/18',   '2025-01-19',   '2025-13-01',   '2025-01-21',
            nan,   '2025-01-23', 'not recorded',   '2025-01-25',
   '2025-01-26',   '2025-01-27',   '2025-01-28',   '2025-01-29',
   '2025-01-30',   '2025-01-31',   '2025-02-01',   '2025-02-02',
   '2025-02-05',   '2025-02-07']
Length: 26, dtype: str

In [15]:
# Default pd.to_datetime fails on 4 rows — let's see which ones
parsed_naive = pd.to_datetime(df['survey_date'], errors='coerce')
df[parsed_naive.isna()][['hh_id', 'survey_date']]

,hh_id,survey_date
6,HH0007,03/15/2025
9,HH0010,2025/01/18
11,HH0012,2025-13-01
12,HH0012,2025-13-01
14,HH0014,NaN
16,HH0016,not recorded


Three values fail because of format inconsistencies; one is not a date at all:\n\n| Raw value | Problem | Fix |\n|---|---|---|\n| `'not recorded'` | Not a date | → `NaN` |\n| `'03/15/2025'` | MM/DD/YYYY format | → `'2025-03-15'` |\n| `'2025/01/18'` | Slash separator | → `'2025-01-18'` |\n| `'2025-13-01'` | Day/month inverted | → `'2025-01-13'` |\n\nThe simplest fix: correct each known bad value explicitly with `.replace()`, then parse.

In [16]:
date_fixes = {
    'not recorded': np.nan,
    '03/15/2025':   '2025-03-15',
    '2025/01/18':   '2025-01-18',
    '2025-13-01':   '2025-01-13',
}

df['survey_date_fixed'] = df['survey_date'].replace(date_fixes)
# Verify the replacements
df[df['survey_date'].isin(date_fixes)][['hh_id', 'survey_date', 'survey_date_fixed']]

,hh_id,survey_date,survey_date_fixed
6,HH0007,03/15/2025,2025-03-15
9,HH0010,2025/01/18,2025-01-18
11,HH0012,2025-13-01,2025-01-13
12,HH0012,2025-13-01,2025-01-13
16,HH0016,not recorded,NaN


In [18]:
df['survey_date_parsed'] = pd.to_datetime(df['survey_date_fixed'], errors='coerce')
# Only the original NaN row and 'not recorded' should remain NaT
df[df['survey_date_parsed'].isna()][['hh_id', 'survey_date']]

,hh_id,survey_date
14,HH0014,NaN
16,HH0016,not recorded


In [19]:
df['survey_month'] = df['survey_date_parsed'].dt.month
df['survey_month'].value_counts(dropna=False).sort_index()

survey_month
1.00    21
2.00     4
3.00     1
NaN      2
Name: count, dtype: int64

**Answers:**\n\n- Four raw values required attention: `'not recorded'` (not a date), `'03/15/2025'` (MM/DD/YYYY — wrong field order), `'2025/01/18'` (slash separator — pandas infers the format from the first rows and rejects inconsistent ones), and `'2025-13-01'` (day and month inverted — month 13 does not exist, the correct date is 2025-01-13). After the explicit fixes, only the original `NaN` row and `'not recorded'` remain `NaT`.\n- `NaT` (Not a Time) is the datetime equivalent of `NaN`. It signals a missing or unparseable date. It behaves like `NaN` in aggregations (skipped by default) but is specific to datetime columns — you cannot store `NaT` in a numeric column.\n- Other useful `.dt` properties: `.dt.year`, `.dt.day`, `.dt.day_name()` (e.g. `'Monday'`), `.dt.quarter`, `.dt.is_month_end`.

**Advanced note — handling unknown formats programmatically (regex)**\n\nThe explicit dictionary works well when the set of bad values is small and known. In a real survey with thousands of enumerators, you may encounter format variants you haven't seen yet. A function using regular expressions can detect and fix patterns automatically, without needing to list every bad value by hand. You will encounter regex later in the course; this is just a preview of what becomes possible.

In [24]:
import re

def standardise_date(value):
    """Normalise any survey_date string to YYYY-MM-DD, or return NaT."""
    if pd.isna(value) or str(value).strip() == 'not recorded':
        return pd.NaT

    s = str(value).strip()

    # MM/DD/YYYY  e.g. 03/15/2025
    m = re.match(r'^([0-9]{2})/([0-9]{2})/([0-9]{4})$', s)
    if m:
        month, day, year = m.groups()
        s = f'{year}-{month}-{day}'

    # YYYY/MM/DD  e.g. 2025/01/18
    m = re.match(r'^([0-9]{4})/([0-9]{2})/([0-9]{2})$', s)
    if m:
        year, month, day = m.groups()
        s = f'{year}-{month}-{day}'

    # YYYY-DD-MM where the middle part > 12 (cannot be a valid month)
    # e.g. 2025-13-01 → 2025-01-13
    m = re.match(r'^([0-9]{4})-([0-9]{2})-([0-9]{2})$', s)
    if m:
        year, part2, part3 = m.groups()
        if int(part2) > 12:
            s = f'{year}-{part3}-{part2}'

    return pd.to_datetime(s, errors='coerce')


for val in ['03/15/2025', '2025/01/18', '2025-13-01', 'not recorded', '2025-01-10']:
    print(f'  {val!r:20s} → {standardise_date(val)}')

  '03/15/2025'         → 2025-03-15 00:00:00
  '2025/01/18'         → 2025-01-18 00:00:00
  '2025-13-01'         → 2025-01-13 00:00:00
  'not recorded'       → NaT
  '2025-01-10'         → 2025-01-10 00:00:00


---

## Task 5 — Row-wise `apply(axis=1)`

In [25]:
def compute_income_per_capita(row):
    """Return income per household member, or NaN if inputs are invalid."""
    income  = row['income_dkw_num']
    hh_size = row['hh_size']

    if pd.isna(income):
        return np.nan
    if pd.isna(hh_size) or hh_size <= 0:
        return np.nan
    return round(income / hh_size, 2)


df['income_per_capita'] = df.apply(compute_income_per_capita, axis=1)

df[['hh_id', 'hh_size', 'income_dkw_num', 'income_per_capita']].head(10)

,hh_id,hh_size,income_dkw_num,income_per_capita
0,HH0001,4,"45,000.00","11,250.00"
1,HH0002,6,"32,000.00","5,333.33"
2,HH0003,3,"54,000.00","18,000.00"
3,HH0004,5,NaN,NaN
4,HH0005,2,NaN,NaN
5,HH0006,7,"-5,000.00",-714.29
6,HH0007,1,"120,000.00","120,000.00"
7,HH0008,0,"38,000.00",NaN
8,HH0009,8,"1,200,000.00","150,000.00"
9,HH0010,4,"41,000.00","10,250.00"


In [26]:
df['suspicious'] = df.apply(
    lambda row: (
        pd.notna(row['income_dkw_num']) and row['income_dkw_num'] > 500_000
    ) or (
        row['hh_size'] <= 0
    ),
    axis=1
)

print(f"{df['suspicious'].sum()} suspicious rows found:")
df[df['suspicious']][['hh_id', 'hh_size', 'income_dkw', 'income_dkw_num']]

4 suspicious rows found:


,hh_id,hh_size,income_dkw,income_dkw_num
7,HH0008,0,38000,"38,000.00"
8,HH0009,8,"1,200,000","1,200,000.00"
10,HH0011,-1,29000,"29,000.00"
22,HH0022,4,999999,"999,999.00"


**Answers:**

- Without `pd.notna()`, comparing `NaN > 500_000` returns `False` silently — pandas does not raise an error. That means rows with missing income would pass the check unnoticed. The `notna()` guard makes the intent explicit and correct.
- Use a **named function** when the logic is non-trivial, needs a docstring, or will be reused elsewhere. Use a **lambda** for short, one-off expressions that are clear enough without a name. The flag here could go either way; the per-capita calculation is complex enough to warrant a named function.
- Flagging before dropping gives you visibility: you can inspect flagged rows, discuss them with the data owner, and document your decision. Dropping immediately is irreversible and leaves no audit trail.

---

## Task 6 — Save to `10_cleaned/`

In [28]:
COLS_OUT = [
    'hh_id',               # identifier — always keep
    'region_code',         # cleaned (leading zeros preserved, '99' replaced)
    'province_name',
    'district',
    'urban_rural',         # cleaned ('Urbn' fixed)
    'hh_size',
    'income_dkw_num',      # cleaned numeric income — drop raw 'income_dkw' and 'income_dkw_clean'
    'survey_date_parsed',  # parsed datetime — drop raw 'survey_date'
    'survey_month',        # derived
    'pop_density',
    'education_code',
    'age',
    'income_per_capita',   # derived
    'suspicious',          # flag — keep so downstream steps can filter or review
]

df_clean = df[COLS_OUT].copy()
print('Shape:', df_clean.shape)
df_clean.head()

Shape: (28, 14)


,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw_num,survey_date_parsed,survey_month,pop_density,education_code,age,income_per_capita,suspicious
0,HH0001,01,Eastern Province,Kurtosis Bay,Urban,4,"45,000.00",2025-01-10,1.00,780,3,42,"11,250.00",False
1,HH0002,02,Northern Province,Vector Hills,Rural,6,"32,000.00",2025-01-11,1.00,120,2,39,"5,333.33",False
2,HH0003,03,Central Province,Polaris District,Urban,3,"54,000.00",2025-01-12,1.00,640,4,33,"18,000.00",False
3,HH0004,04,Southern Province,Lagoon Point,Rural,5,NaN,2025-01-13,1.00,80,1,51,NaN,False
4,HH0005,05,Western Province,Gamma Plains,Urban,2,NaN,2025-01-13,1.00,520,2,28,NaN,False


In [29]:
DATA_CLEAN_DIR = '../../../data/10_cleaned'
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)

output_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

df_clean.to_csv(output_path, index=False)

print('Saved to:', output_path)

Saved to: ../../../data/10_cleaned/datania_households_clean.csv


In [30]:
df_check = pd.read_csv(output_path)
print('Reloaded shape:', df_check.shape)
print()
print(df_check.dtypes)
df_check.head()

Reloaded shape: (28, 14)

hh_id                     str
region_code           float64
province_name             str
district                  str
urban_rural               str
hh_size                 int64
income_dkw_num        float64
survey_date_parsed        str
survey_month          float64
pop_density             int64
education_code          int64
age                     int64
income_per_capita     float64
suspicious               bool
dtype: object


,hh_id,region_code,province_name,district,urban_rural,hh_size,income_dkw_num,survey_date_parsed,survey_month,pop_density,education_code,age,income_per_capita,suspicious
0,HH0001,1.00,Eastern Province,Kurtosis Bay,Urban,4,"45,000.00",2025-01-10,1.00,780,3,42,"11,250.00",False
1,HH0002,2.00,Northern Province,Vector Hills,Rural,6,"32,000.00",2025-01-11,1.00,120,2,39,"5,333.33",False
2,HH0003,3.00,Central Province,Polaris District,Urban,3,"54,000.00",2025-01-12,1.00,640,4,33,"18,000.00",False
3,HH0004,4.00,Southern Province,Lagoon Point,Rural,5,NaN,2025-01-13,1.00,80,1,51,NaN,False
4,HH0005,5.00,Western Province,Gamma Plains,Urban,2,NaN,2025-01-13,1.00,520,2,28,NaN,False


**Answers:**

- Without `index=False`, pandas writes the integer row index (0, 1, 2, …) as an extra first column called `Unnamed: 0` when the file is reloaded. It adds no information and clutters the dataset.
- After reloading, `survey_date_parsed` comes back as `object` (text). CSV stores everything as plain text — there is no way to encode a datetime type in a CSV file. This means anyone loading the cleaned file must convert the date column again. For long-lived data files, formats like Parquet or Feather preserve types.
- The `suspicious` column should be **kept** in the cleaned output. Downstream steps (e.g. exclusion rules, manual review) need to know which rows were flagged. Dropping it would lose that information. If the output is meant for external stakeholders who don't need internal flags, it could be excluded in a separate reporting output.